# Compute resources

This notebook is for when a global analysis keeps running out of memory. We explain why the amount of data a global request reads can't be reduced but the memory it needs can, check how big a global request is, and show how we ran one on a small cloud machine.

For the short version, see Section 7 of [`subsetting-and-exporting.ipynb`](subsetting-and-exporting.ipynb). If you come across a term you don't know, check the [glossary](https://github.com/carbonplan/sai-downscaling-data-utils/blob/main/GLOSSARY.md).

**Contents:**
1. Reads versus memory
2. How much a global request reads
3. Running it on a small machine
4. Measured results

## Setup

Follow the [installation instructions](https://github.com/carbonplan/sai-downscaling-data-utils#installation) in the README, then launch JupyterLab with `pixi run jupyter lab`. The cell below loads helper functions from the [`scripts/`](../scripts/README.md) folder, so open this notebook from inside the cloned repository.

In [1]:
import sys
from pathlib import Path

import flox  # noqa - xarray groupby speedup
import numpy as np
import xarray as xr

# The helper functions live in the repository's scripts/ folder.
repo = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "data_access.py").exists()), None)
if repo is None:
    raise RuntimeError("Open this notebook from inside the sai-downscaling-data-utils repository.")
sys.path.insert(0, str(repo / "scripts"))

from notebook_helpers import first_decade  # noqa: E402
from data_access import check_members, describe_request, load_downscaling_store  # noqa: E402

## 1. Reads versus memory

Analyses that cover the whole globe are where notebooks most often crash, because they ask for so much data. Moving to a bigger machine usually doesn't help, because memory isn't what's holding you back.

The amount of data a request reads is fixed. Data is stored in chunks, and every chunk your selection touches has to be downloaded and decompressed in full. Two variables over the whole globe for a decade read about 35 GB no matter what machine you use. Since each chunk holds about a year of data, the read grows with the time span you ask for: a single global day reads 1.6 GB, and a decade reads about 17.5 GB per variable.

What you can change is how much of that data you hold in memory at once:

| Code | What it holds in memory |
|---|---|
| `ds["tas"].sel(time=decade).load()` | All 15.2 GB of the decade at once, before anything is reduced (30.3 GB for `tas` and `pr` together). This is the most common cause of a crashed notebook. |
| `ds["tas"].sel(time=decade).resample(time="YE").mean().compute()` | A few 3.8 MB chunks at a time while it works, and then just the 41.5 MB of annual means it returns. On a 4 GB machine the full two-variable run used at most 1.88 GB (Section 4). |

You may also come across `chunks="auto"`. It isn't an alternative to reducing your data first: it's a setting for how data is opened, and it makes each piece of work much bigger (about 102 MB instead of 3.8 MB) without reading any less. `load_downscaling_store(...)` already opens the data with the right chunk size, so you don't need to change anything.

In short, a global analysis is limited by how fast the data can be read, not by how much memory you have. It will run on a laptop; it just takes as long as the reads take.

## 2. How much a global request reads

The cell below checks how much data the global request in this notebook reads: `tas` and `pr` for one decade. Nothing is computed, so it finishes right away.

In [2]:
# ===== CUSTOMIZE: Choose the data for the global run =====
gcm_name = "CESM2-WACCM6"  # "CESM2-WACCM6" or "UKESM1-1-LL"
method_name = "bcsd"  # "bcsd" or "qdmsd"
scenario_name = "ssp245"  # "historical", "ssp245", "g6_1p5k" or "g6_1p5k_end"
member_name = None  # None takes the pinned default
product_name = "downscaled"  # or "debiased_coarse"
global_vars = ["tas", "pr"]
# ==========================================================

# No modification needed below
# Check the pairing first: tas+pr share a member, tas+tasmax would not.
check_members(scenario_name, global_vars, gcm=gcm_name, method=method_name, product=product_name)
print()

total_read = 0
for v in global_vars:
    ds_v = load_downscaling_store(
        scenario_name, v, gcm=gcm_name, method=method_name, member=member_name,
        product=product_name,
    )
    total_read += describe_request(ds_v[v].sel(time=first_decade(ds_v)), f"Global, 10 yr, {v}")

print(f"\ncombined read for both variables: {total_read / 1e9:.1f} GB")
print("This is the floor. Reducing changes memory, not bytes read.")

CESM2-WACCM6/ssp245: tas, pr all share member 003



Global, 10 yr, tas:
  shape          {'time': 3653, 'lat': 721, 'lon': 1440}
  logical size     15.171 GB
  chunks touched     4620  (3.8 MB each, store chunks)
  data read        17.484 GB


Global, 10 yr, pr:
  shape          {'time': 3653, 'lat': 721, 'lon': 1440}
  logical size     15.171 GB
  chunks touched     4620  (3.8 MB each, store chunks)
  data read        17.484 GB

combined read for both variables: 35.0 GB
This is the floor. Reducing changes memory, not bytes read.


## 3. Running it on a small machine

The cell below reduces the data before computing anything, so the only thing held in memory at the end is the annual, area-weighted result. It's turned off by default because it reads about 35 GB.

To try it on a small machine, we recommend a cloud VM in `us-west-2`, the same region as the data, so the downloads stay within the region instead of going over the internet to your laptop:

```bash
pixi run -e cloud coiled run \
    --vm-type c7i.large --region us-west-2 \
    -- env RUN_GLOBAL_DEMO=1 python your_runner.py
```

A `c7i.large` has 2 CPUs and 4 GB of memory, which is smaller than most laptops. We deliberately didn't use a `t3` instance: those slow down once their CPU credits run out, so the timing would measure that rather than the approach itself.

To work interactively on the same size of machine, `coiled notebook start --vm-type c7i.large --region us-west-2 --sync` opens JupyterLab there with your local files.

> **Be careful with `--sync`.** It syncs in both directions, so files that exist on your computer but not on the remote machine can be deleted from your working directory, including files git isn't tracking and can't restore. Commit or back up that work first. The benchmark command above leaves out `--sync` on purpose, since a self-contained script doesn't need anything synced.

The results we measured are in Section 4.

In [3]:
# ===== CUSTOMIZE: opt in to the full global run =====
# Flip this to True, or set the RUN_GLOBAL_DEMO=1 environment variable -- which
# is how the measured numbers in Section 4 were produced on a remote VM in one command.
# Either way it reads about 35 GB, so it is off by default.
import os

RUN_GLOBAL_DEMO = os.environ.get("RUN_GLOBAL_DEMO", "0").lower() in ("1", "true", "yes")
# =====================================================

if RUN_GLOBAL_DEMO:
    import resource
    import sys
    import time

    t0 = time.perf_counter()
    reductions = {}
    for v in global_vars:
        ds_v = load_downscaling_store(
            scenario_name, v, gcm=gcm_name, method=method_name, member=member_name,
            product=product_name,
        )
        da_v = ds_v[v].sel(time=first_decade(ds_v))
        # Reduce FIRST. Annual means collapse ~3,653 daily steps into 10, and the
        # area-weighted spatial mean collapses the grid to a single number per year.
        annual_v = da_v.resample(time="YE").mean()
        weights_v = np.cos(np.deg2rad(da_v.lat))
        reductions[v] = annual_v.weighted(weights_v).mean(dim=["lat", "lon"])

    # One graph for both variables, so each is streamed through once.
    result = xr.Dataset(reductions).compute()

    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_gb = peak / 1024**3 if sys.platform == "darwin" else peak / 1024**2

    print(result)
    print(f"\npeak RSS   {peak_gb:6.2f} GB")
    print(f"wall time  {time.perf_counter() - t0:6.0f} s")
    print(f"result     {result.nbytes} bytes held, from {total_read / 1e9:.1f} GB read")
else:
    print("RUN_GLOBAL_DEMO is False - skipping the ~35 GB global run.")
    print("Measured results from a 4 GB VM are recorded in Section 4.")


RUN_GLOBAL_DEMO is False - skipping the ~35 GB global run.
Measured results from a 4 GB VM are recorded in Section 4.


## 4. Measured results

We ran the cell above with `RUN_GLOBAL_DEMO=1` on a single `c7i.large` (2 CPUs, 4 GB of memory) in `us-west-2`:

| Metric | Value |
|---|---|
| Variables | `tas`, `pr` (CESM2-WACCM6, bcsd, ssp245, member `003`) |
| Window | 2015-2024, global (`time=3653, lat=721, lon=1440`) |
| Logical size | 30.3 GB |
| Data read | 35.0 GB |
| **Peak memory** | **1.88 GB** |
| Wall time | 829 s (~14 min) |
| Result held | 160 bytes |

The run read 35 GB of data on a machine with only 4 GB of memory, and it never used more than 1.88 GB. That's because only a handful of 3.8 MB chunks are in memory at any one time, so the memory you need depends on the chunk size and how many chunks are processed at once, not on how much data you read. Calling `.load()` on the same data would need more than 30 GB of free memory.

The trade-off is time. The run took about 14 minutes, reading roughly 42 MB per second, and it was limited by decompressing data on 2 CPUs rather than by memory. More CPUs would make it faster; more memory would not.

If you need to keep a map rather than a single number per year, reduce along time first. For example, `resample(time="YE").mean()` on a global decade gives 41.5 MB, which is easy to save to Zarr. Calling `.load()` on the full 30.3 GB is not.